In [26]:
import pandas as pd
DATA_PATH = 'jobpostingdata.csv'
MODEL_PATH = 'model.pkl'

df = pd.read_csv(DATA_PATH)

In [27]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [28]:
def get_tag(tag: str):
    if tag[0] in ['V', 'N', 'R']:
        return tag[0].lower()
    elif tag[0] == 'J':
        return 'a'
    else:
        return 'n'

def lemmatization(tokens):
    lemmatizer = WordNetLemmatizer()
    tagged = pos_tag(tokens)
    lemmatized = []
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, get_tag(tag))
        lemmatized.append(result)
    return lemmatized

def text_preprocessor(sentence: str):
    eng_stopwords = stopwords.words('english')
    tokens = word_tokenize(sentence)
    cleaned = [
        token.lower() for token in tokens if token not in eng_stopwords
        and
        token.isalpha()
    ]
    lemmatized = lemmatization(cleaned)
    return lemmatized

In [29]:
def feature_extraction(sentence, existed_words):
    unique_words = set(text_preprocessor(sentence))
    feature = {}
    for word in existed_words:
        feature[word] = (word in unique_words)
    return feature

In [30]:
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify import accuracy
from sklearn.model_selection import train_test_split

In [31]:
def train_classifier():
    x = df['text']
    y = df['fraudulent']

    corpus = ' '.join(x)
    existed_words = set(text_preprocessor(corpus))

    features = [
        (feature_extraction(sentence, existed_words), category)
        for sentence, category in zip(x, y)
    ]

    train_data, test_data = train_test_split(
        features, test_size=0.2, random_state=42
    )

    classifier = NaiveBayesClassifier.train(train_data)
    acc = accuracy(classifier, test_data)
    print(acc)
    return classifier, existed_words

In [32]:
train_classifier()

0.75


(<nltk.classify.naivebayes.NaiveBayesClassifier at 0x1989b8351d0>,
 {'anaheim',
  'carol',
  'plusprevious',
  'economist',
  'exp',
  'nonprofit',
  'examination',
  'patio',
  'dojo',
  'cashier',
  'confidentialityexcellent',
  'hacker',
  'upon',
  'hayward',
  'rfis',
  'function',
  'tasksresponsibilites',
  'initiativeability',
  'dermatologyrelocation',
  'pass',
  'transaction',
  'easygoing',
  'entrymanaging',
  'twilio',
  'quo',
  'zu',
  'compassion',
  'coursework',
  'school',
  'deduction',
  'responsibility',
  'although',
  'integrated',
  'tx',
  'toprovide',
  'professionallyinformal',
  'pipe',
  'resolve',
  'imaginative',
  'inconsistency',
  'sloane',
  'subnetting',
  'freebsd',
  'degreeextra',
  'inout',
  'createddrive',
  'programmeorganizing',
  'weld',
  'incomeother',
  'englishhow',
  'northern',
  'teams',
  'experienceworking',
  'διαχείρισης',
  'neededwork',
  'legit',
  'korea',
  'dhcp',
  'όλους',
  'clearlyable',
  'respond',
  'importantly',
 

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def train_vectorizer():
    x = df['text']
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english'
    )
    vectors = vectorizer.fit_transform(x)
    return vectorizer, vectors

In [5]:
import os
import pickle

In [2]:
def load_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, 'rb') as f:
            classifier, existed_words, vectorizer, vectors = pickle.load(f)
        return classifier, existed_words, vectorizer, vectors
    else:
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()

        with open(MODEL_PATH, 'wb') as f:
            pickle.dump((classifier, existed_words, vectorizer, vectors), f)

In [8]:
def print_menu(sentence, cateogory):
    print(sentence)
    print(cateogory)
    print("1. Write Text")
    print("2. Recommend")
    print("3. NER")
    print("4. Exit")

In [ ]:
def write_text():
    sent = ''
    while len(sent.strip()) < 20 or len(sent.strip().split) < 3:
        sent = input("Write a sentence: ")
    return sent

In [4]:
def classify_text(classifier, sentence, existed_words):
    feature = feature_extraction(sentence, existed_words)
    category = classifier.classify(feature)

    if category == 1:
        return "TRUE"
    else:
        return "FAKE"

In [5]:
def load_nlp(nlp, sentence):
    doc = nlp(sentence)
    ents_dicts = {}

    for ent in doc.ents:
        if ent.label_ not in ents_dicts.keys():
            ents_dicts[ent.label_] = []
        ents_dicts[ent.label_].append(ent.text)
    return ents_dicts

In [6]:
def print_NER(ents_dicts):
    if ents_dicts:
        for key, value in ents_dicts.items():
            print(f"{key}: {value}")
    return

In [1]:
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
def recommend_top_n(vectorizer: TfidfVectorizer, job_vectors, query, topn=5):
    vectorized_query = vectorizer.transform([query])
    similarity = cosine_similarity(job_vectors, vectorized_query).flatten()
    topidx = similarity.argsort()[::-1][:topn]

    for i, idx in enumerate(topidx, 1):
        print(f"{i}. {df['title'].iloc[idx]}")

In [6]:
import spacy